# Employee Performance Prediction
## INX Future Inc — Employee Performance
### Made by - **Chinmay Shastry**
**Goal:** Use the trained model to predict performance for new or existing employees.

## Step 1 — Load Model & Libraries

In [5]:
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

with open('../../data/processed/trained_model.pkl', 'rb') as f:
    art = pickle.load(f)

model    = art['model']
scaler   = art['scaler']
features = art['selected_features']
top3     = art['top3_features']

print("Model loaded:", type(model).__name__)
print("Top 3 predictive features:", top3)

Model loaded: RandomForestClassifier
Top 3 predictive features: ['EmpLastSalaryHikePercent', 'EmpEnvironmentSatisfaction', 'YearsSinceLastPromotion']


## Step 2 — Predict on Full Dataset

In [6]:
df_proc = pd.read_csv('../../data/processed/employee_performance_processed.csv')
df_raw  = pd.read_csv('../../data/raw/INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.csv')

X      = df_proc[features]
y_true = df_proc['PerformanceRating']

X_scaled = scaler.transform(X)
y_pred   = model.predict(X_scaled)

from sklearn.metrics import accuracy_score
print(f"Accuracy on full dataset: {accuracy_score(y_true, y_pred):.4f}")

Accuracy on full dataset: 0.9717


## Step 3 — Predict a Single New Employee

In [7]:
label_map = {2: 'Low Performer', 3: 'Good Performer', 4: 'Excellent Performer'}

# Create a blank employee record and fill in known values
new_emp = {f: 0 for f in features}
new_emp.update({
    'EmpLastSalaryHikePercent'   : 18,
    'EmpEnvironmentSatisfaction' : 4,
    'EmpJobSatisfaction'         : 4,
    'EmpJobInvolvement'          : 3,
    'TotalWorkExperienceInYears' : 8,
    'TrainingTimesLastYear'      : 3,
    'EmpJobLevel'                : 3,
    'Age'                        : 35,
})

new_df    = pd.DataFrame([new_emp])
new_pred  = model.predict(scaler.transform(new_df))[0]
new_proba = model.predict_proba(scaler.transform(new_df))[0]

print(f"Predicted Rating : {new_pred} — {label_map[new_pred]}")
print("\nClass Probabilities:")
for cls, prob in zip(model.classes_, new_proba):
    print(f"  Rating {cls} ({label_map[cls]}): {prob:.2%}")

Predicted Rating : 3 — Good Performer

Class Probabilities:
  Rating 2 (Low Performer): 3.00%
  Rating 3 (Good Performer): 83.00%
  Rating 4 (Excellent Performer): 14.00%


## Step 4 — Export Batch Predictions

In [8]:
df_out = df_raw[['EmpNumber','EmpDepartment','EmpJobRole','PerformanceRating']].copy()
df_out.rename(columns={'PerformanceRating': 'Actual_Rating'}, inplace=True)
df_out['Predicted_Rating']    = y_pred
df_out['Performance_Label']   = df_out['Predicted_Rating'].map(label_map)
df_out['Correct_Prediction']  = df_out['Actual_Rating'] == df_out['Predicted_Rating']

df_out.to_csv('../../data/processed/employee_predictions.csv', index=False)
print("Predictions saved!")
print(f"Overall Accuracy: {df_out['Correct_Prediction'].mean():.2%}")
print("\nSample:")
print(df_out.head(10).to_string(index=False))

Predictions saved!
Overall Accuracy: 97.17%

Sample:
EmpNumber   EmpDepartment           EmpJobRole  Actual_Rating  Predicted_Rating   Performance_Label  Correct_Prediction
 E1001000           Sales      Sales Executive              3                 3      Good Performer                True
 E1001006           Sales      Sales Executive              3                 3      Good Performer                True
 E1001007           Sales      Sales Executive              4                 4 Excellent Performer                True
 E1001009 Human Resources              Manager              3                 3      Good Performer                True
 E1001010           Sales      Sales Executive              3                 3      Good Performer                True
 E1001011     Development            Developer              4                 4 Excellent Performer                True
 E1001016           Sales Sales Representative              3                 3      Good Performer        